# CLEAR: Zero-Shot Tennis Ball & Player Tracker + Virtual Clay-Court Line Judge

**Course:** Computer Vision and Deep Learning — Six-Week Capstone Project (Advent 2026)  
**Group 1 (UCU):** Andrea NABUUMA (B28112), Robert WANYAMA, Blair TONNY (B30009), Nathaniel MUGENYI (B28313)

---

### What This Notebook Does (Self-Contained for Google Colab & Local `uv`)
1. **Anchor-Calibrated Virtual Court (`H`, `H^-1`)**: Projects regulation ITF court boundaries (`23.77m x 10.97m`) + a `±15 cm` **Doubt Zone** onto clay courts where painted lines are worn or rain-wiped.
2. **Zero-Shot Hybrid Detection**: Combines pretrained COCO **YOLOv8** (`person`, `sports ball`) with a **3-Frame Temporal Motion + Clay HSV Segmentation** branch to catch fast, motion-blurred 4–10px tennis balls on low-contrast red clay.
3. **Multi-Object Tracking**:
   - **Ball**: 4-state Constant-Velocity **Kalman Filter** with Mahalanobis/Euclidean gating, coasting across occlusions, and speed estimation (`km/h`).
   - **Players**: **ByteTrack-lite** two-stage association for Near (`P1`) and Far (`P2`) players, projecting foot contact points to court metres for distance covered (`m`), sprint speed (`km/h`), and court-zone occupancy.
4. **Physics-Impulse Bounce Judge**: Cancels camera perspective slope via second-order vertical impulse $\kappa(t) = v(t) - \frac{1}{2}(v(t-2) + v(t+2))$ + piecewise linear/parabolic ray intersection for sub-pixel bounce localization.
5. **Broadcast HUD + 2D Court Radar + Score Engine**: Draws virtual court lines, ball comet trail, player tactical boxes, a live top-down 2D court radar, Hawk-Eye zoom inset on close calls, and automatic `0/15/30/40/Deuce/Ad` tennis scoring.

In [ ]:
# @title 1. Install Dependencies (Colab / Local)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q ultralytics opencv-python-headless numpy scipy matplotlib pyyaml

import json
from collections import deque
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Deque, Dict, List, Literal, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np

print(f"OpenCV: {cv2.__version__} | NumPy: {np.__version__} | Running in Colab: {IN_COLAB}")

In [ ]:
# @title 2. Hyperparameter Control Panel (Tweak Here!)

# --- Court & Calibration Settings ---
COURT_MODE = "singles"  # @param ["singles", "doubles"]
LINE_CONDITION = "worn"  # @param ["fresh", "worn", "wiped"]
DOUBT_ZONE_CM = 15.0  # @param {type:"slider", min:5.0, max:30.0, step:1.0}
BALL_CONTACT_RADIUS_CM = 3.8  # @param {type:"number"}

# --- Detector Settings (Zero-Shot YOLO + 3-Frame Temporal Clay HSV) ---
DETECTOR_MODE = "hybrid"  # @param ["hybrid", "yolo", "classical"]
YOLO_WEIGHTS = "yolov8n.pt"  # @param ["yolov8n.pt", "yolov8s.pt", "yolo11n.pt"]
BALL_CONF_THRESH = 0.20  # @param {type:"slider", min:0.05, max:0.60, step:0.05}
PLAYER_CONF_THRESH = 0.35  # @param {type:"slider", min:0.15, max:0.75, step:0.05}
USE_3FRAME_TEMPORAL = True  # @param {type:"boolean"}

# HSV bounds for optic-yellow tennis ball on red/ochre clay
BALL_HSV_LOWER = (24, 95, 140)
BALL_HSV_UPPER = (46, 255, 255)
MIN_BALL_RADIUS_PX = 2
MAX_BALL_RADIUS_PX = 18

# --- Kalman Tracker & Bounce Judge Settings ---
VIDEO_FPS = 60.0
KALMAN_PROCESS_NOISE = 1.8  # @param {type:"number"}
KALMAN_MEASUREMENT_NOISE = 2.5  # @param {type:"number"}
KALMAN_GATE_DIST_PX = 95.0  # @param {type:"number"}
MIN_BOUNCE_IMPULSE_PX = 1.6  # @param {type:"slider", min:0.8, max:4.0, step:0.1}
USE_PARABOLIC_REFINEMENT = True  # @param {type:"boolean"}

print("Hyperparameters loaded successfully.")

In [ ]:
# @title 3. ITF Court Geometry, Anchor Homography & Virtual Line Renderer

@dataclass(frozen=True)
class ITFCourtSpec:
    length_m: float = 23.77
    doubles_width_m: float = 10.97
    singles_width_m: float = 8.23
    alley_width_m: float = 1.37
    net_y_m: float = 11.885
    far_service_y_m: float = 5.485
    near_service_y_m: float = 18.285
    center_x_m: float = 5.485
    center_mark_len_m: float = 0.15
    line_width_m: float = 0.05

SPEC = ITFCourtSpec()

ANCHOR_COORDS_M: Dict[str, Tuple[float, float]] = {
    "far_left_doubles": (0.0, 0.0),
    "far_right_doubles": (SPEC.doubles_width_m, 0.0),
    "net_right_sideline": (SPEC.doubles_width_m, SPEC.net_y_m),
    "near_right_doubles": (SPEC.doubles_width_m, SPEC.length_m),
    "near_left_doubles": (0.0, SPEC.length_m),
    "net_left_sideline": (0.0, SPEC.net_y_m),
}

COURT_LINES_M = [
    ("far_baseline", (0.0, 0.0), (SPEC.doubles_width_m, 0.0), "baseline"),
    ("near_baseline", (0.0, SPEC.length_m), (SPEC.doubles_width_m, SPEC.length_m), "baseline"),
    ("left_doubles_sideline", (0.0, 0.0), (0.0, SPEC.length_m), "doubles"),
    ("right_doubles_sideline", (SPEC.doubles_width_m, 0.0), (SPEC.doubles_width_m, SPEC.length_m), "doubles"),
    ("left_singles_sideline", (SPEC.alley_width_m, 0.0), (SPEC.alley_width_m, SPEC.length_m), "singles"),
    ("right_singles_sideline", (SPEC.doubles_width_m - SPEC.alley_width_m, 0.0), (SPEC.doubles_width_m - SPEC.alley_width_m, SPEC.length_m), "singles"),
    ("far_service_line", (SPEC.alley_width_m, SPEC.far_service_y_m), (SPEC.doubles_width_m - SPEC.alley_width_m, SPEC.far_service_y_m), "service"),
    ("near_service_line", (SPEC.alley_width_m, SPEC.near_service_y_m), (SPEC.doubles_width_m - SPEC.alley_width_m, SPEC.near_service_y_m), "service"),
    ("center_service_line", (SPEC.center_x_m, SPEC.far_service_y_m), (SPEC.center_x_m, SPEC.near_service_y_m), "service"),
    ("net_line", (0.0, SPEC.net_y_m), (SPEC.doubles_width_m, SPEC.net_y_m), "net"),
]

def evaluate_court_point(
    x_m: float,
    y_m: float,
    mode: str = "singles",
    target_half: str = "full",
    ball_contact_radius_cm: float = 3.8,
    doubt_zone_cm: float = 15.0,
    reproj_uncertainty_cm: float = 3.0,
) -> Dict[str, object]:
    x_min = 0.0 if mode == "doubles" else SPEC.alley_width_m
    x_max = SPEC.doubles_width_m if mode == "doubles" else (SPEC.doubles_width_m - SPEC.alley_width_m)
    left_name = "left_doubles_sideline" if mode == "doubles" else "left_singles_sideline"
    right_name = "right_doubles_sideline" if mode == "doubles" else "right_singles_sideline"

    if target_half == "far":
        y_min, y_max = 0.0, SPEC.net_y_m
        top_name, bottom_name = "far_baseline", "net_line"
    elif target_half == "near":
        y_min, y_max = SPEC.net_y_m, SPEC.length_m
        top_name, bottom_name = "net_line", "near_baseline"
    else:
        y_min, y_max = 0.0, SPEC.length_m
        top_name, bottom_name = "far_baseline", "near_baseline"

    contact_m = (ball_contact_radius_cm / 100.0) + SPEC.line_width_m * 0.5
    candidates = [
        ((x_m - x_min) + contact_m, left_name),
        ((x_max - x_m) + contact_m, right_name),
        ((y_m - y_min) + contact_m, top_name),
        ((y_max - y_m) + contact_m, bottom_name),
    ]
    min_dist_m, closest_line = min(candidates, key=lambda item: item[0])
    signed_margin_cm = float(min_dist_m * 100.0)
    abs_margin_cm = abs(signed_margin_cm)
    is_in = signed_margin_cm >= 0.0
    conf = float(0.50 + 0.49 * np.tanh(0.65 * (abs_margin_cm / max(1.5, reproj_uncertainty_cm))))
    return {
        "x_m": round(float(x_m), 3),
        "y_m": round(float(y_m), 3),
        "is_in": is_in,
        "call": "IN" if is_in else "OUT",
        "signed_margin_cm": round(signed_margin_cm, 1),
        "abs_margin_cm": round(abs_margin_cm, 1),
        "closest_line": closest_line,
        "in_doubt_zone": abs_margin_cm <= doubt_zone_cm,
        "confidence": round(conf, 3),
    }

class CourtCalibrator:
    def __init__(self, anchors_px: Optional[Dict[str, Tuple[float, float]]] = None):
        self.anchors_px = anchors_px or {}
        self.H_court_to_img: Optional[np.ndarray] = None
        self.H_img_to_court: Optional[np.ndarray] = None
        self.reproj_error_cm: float = 2.8
        self.method: str = "anchors_6"
        if len(self.anchors_px) >= 4:
            self.fit_from_anchors(self.anchors_px)

    def fit_from_anchors(self, anchors_px: Dict[str, Tuple[float, float]]) -> None:
        names = [k for k in anchors_px if k in ANCHOR_COORDS_M]
        pts_m = np.array([ANCHOR_COORDS_M[k] for k in names], dtype=np.float64)
        pts_px = np.array([anchors_px[k] for k in names], dtype=np.float64)
        H, _ = cv2.findHomography(pts_m, pts_px, cv2.RANSAC, 5.0)
        self.H_court_to_img = H
        self.H_img_to_court = np.linalg.inv(H)
        proj_m = self.image_to_court(pts_px)
        self.reproj_error_cm = max(1.5, float(np.mean(np.linalg.norm(proj_m - pts_m, axis=1)) * 100.0))

    @classmethod
    def from_default_camera_geometry(cls, width: int = 1280, height: int = 720) -> "CourtCalibrator":
        corners = {
            "far_left_doubles": (width * 0.285, height * 0.215),
            "far_right_doubles": (width * 0.715, height * 0.215),
            "near_right_doubles": (width * 0.910, height * 0.875),
            "near_left_doubles": (width * 0.090, height * 0.875),
        }
        base = cls(corners)
        nl = base.point_court_to_image(0.0, SPEC.net_y_m)
        nr = base.point_court_to_image(SPEC.doubles_width_m, SPEC.net_y_m)
        return cls({**corners, "net_right_sideline": nr, "net_left_sideline": nl})

    def court_to_image(self, pts_m) -> np.ndarray:
        arr = np.asarray(pts_m, dtype=np.float64).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(arr, self.H_court_to_img).reshape(-1, 2)

    def image_to_court(self, pts_px) -> np.ndarray:
        arr = np.asarray(pts_px, dtype=np.float64).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(arr, self.H_img_to_court).reshape(-1, 2)

    def point_image_to_court(self, u: float, v: float) -> Tuple[float, float]:
        p = self.image_to_court([[u, v]])[0]
        return float(p[0]), float(p[1])

    def point_court_to_image(self, xm: float, ym: float) -> Tuple[float, float]:
        p = self.court_to_image([[xm, ym]])[0]
        return float(p[0]), float(p[1])

print("Calibration and ITF Court Geometry ready.")

In [ ]:
# @title 4. Load CLEAR Engine & Generate Synthetic UCU Clay-Court Rally (or Upload Custom Video)
import os
for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/CLEAR/CLEAR"), Path("/content/CLEAR")]:
    if (candidate / "src").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        os.chdir(candidate)
        break

try:
    from src.data.synthetic_rally import generate_synthetic_clay_rally
    from src.pipeline import ClearPipeline
    from src.calibration.homography import CourtCalibrator as RepoCalibrator
    USING_REPO_MODULES = True
except ImportError:
    USING_REPO_MODULES = False

DEMO_VIDEO = Path("data/demo/ucu_clay_rally.mp4")
DEMO_GT = Path("data/demo/ucu_clay_rally_gt.json")
DEMO_CALIB = Path("calib/session.json")

if USING_REPO_MODULES:
    gt_data = generate_synthetic_clay_rally(
        video_path=DEMO_VIDEO,
        gt_json_path=DEMO_GT,
        calib_json_path=DEMO_CALIB,
        line_condition=LINE_CONDITION,
    )
    print(f"Generated clay rally video: {DEMO_VIDEO} ({len(gt_data['frames'])} tracked ball frames, {len(gt_data['bounces'])} bounces)")
else:
    print("Tip: In Colab, mount Drive or upload the CLEAR folder so `src/` is available.")

In [ ]:
# @title 5. Run Zero-Shot Pipeline & Inspect Bounce Calls
if USING_REPO_MODULES:
    calibrator = RepoCalibrator.load(DEMO_CALIB)
    pipeline = ClearPipeline(
        calibrator=calibrator,
        court_mode=COURT_MODE,
        detector_mode=DETECTOR_MODE,
        fps=VIDEO_FPS,
        doubt_zone_cm=DOUBT_ZONE_CM,
        use_3frame_temporal=USE_3FRAME_TEMPORAL,
        use_parabolic_refinement=USE_PARABOLIC_REFINEMENT,
    )
    pipeline.detector.ball_conf_thresh = BALL_CONF_THRESH
    pipeline.detector.player_conf_thresh = PLAYER_CONF_THRESH
    pipeline.detector.ball_hsv_lower = np.array(BALL_HSV_LOWER, dtype=np.uint8)
    pipeline.detector.ball_hsv_upper = np.array(BALL_HSV_UPPER, dtype=np.uint8)
    pipeline.ball_tracker.process_noise = KALMAN_PROCESS_NOISE
    pipeline.ball_tracker.measurement_noise = KALMAN_MEASUREMENT_NOISE
    pipeline.ball_tracker.gate_dist_px = KALMAN_GATE_DIST_PX
    pipeline.bounce_judge.min_bounce_vy_px = MIN_BOUNCE_IMPULSE_PX

    report = pipeline.run_video(DEMO_VIDEO, out_dir="results/notebook_run")
    print(json.dumps(report["line_calling"], indent=2))
    print("\nPlayer Analytics:")
    print(json.dumps(report["players"], indent=2))
    print("\nRunning Score:", report["score"]["score_line"])

In [ ]:
# @title 6. Visualize Bounce Frames (Virtual Court + Hawk-Eye Call Card + 2D Radar)
if USING_REPO_MODULES:
    cap = cv2.VideoCapture(str( Path("results/notebook_run/annotated_rally.mp4") ))
    bounce_frames = [b["frame_idx"] for b in report["line_calling"]["bounces"]]
    fig, axes = plt.subplots(2, 2, figsize=(16, 9), facecolor="#0f172a")
    axes = axes.flatten()
    for idx, b_info in enumerate(report["line_calling"]["bounces"][:4]):
        f_target = int(b_info["frame_idx"]) + 3
        cap.set(cv2.CAP_PROP_POS_FRAMES, f_target)
        ret, frm = cap.read()
        if ret:
            rgb = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(rgb)
            axes[idx].set_title(
                f"Frame #{b_info['frame_idx']} | {b_info['call']} ({b_info['signed_margin_cm']:+.1f} cm vs {b_info['closest_line']})",
                color="#f8fafc", fontsize=11, weight="bold"
            )
            axes[idx].axis("off")
    cap.release()
    plt.tight_layout()
    plt.show()

    # Show 2D Top-Down Court Analytics Diagram
    chart_img = cv2.imread("results/notebook_run/court_analytics.png")
    if chart_img is not None:
        plt.figure(figsize=(13, 8))
        plt.imshow(cv2.cvtColor(chart_img, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()

In [ ]:
# @title 7. Controlled Ablation & Surface Condition Experiments (C2 & C3)
if USING_REPO_MODULES:
    from scripts.evaluate import evaluate_run

    results_table = []
    for surface in ["fresh", "worn", "wiped"]:
        v_path = Path(f"data/demo/rally_{surface}.mp4")
        gt = generate_synthetic_clay_rally(v_path, line_condition=surface)
        cap = cv2.VideoCapture(str(v_path))
        _, first_f = cap.read()
        cap.release()

        # Baseline B2 (Painted lines Hough) + B3 (unrefined vertex)
        b2_cal = RepoCalibrator.from_painted_lines_baseline(first_f)
        p_base = ClearPipeline(calibrator=b2_cal, detector_mode="classical", use_3frame_temporal=False, use_parabolic_refinement=False)
        p_base.run_video(v_path, out_dir=f"results/ablation/base_{surface}")
        m_base = evaluate_run(p_base, gt)

        # CLEAR Final (6-Anchor Virtual Court + 3-Frame Temporal + Parabolic Fit)
        a_cal = RepoCalibrator.from_default_camera_geometry()
        p_clear = ClearPipeline(calibrator=a_cal, detector_mode="hybrid", use_3frame_temporal=True, use_parabolic_refinement=True)
        p_clear.run_video(v_path, out_dir=f"results/ablation/clear_{surface}")
        m_clear = evaluate_run(p_clear, gt)

        results_table.append({
            "surface": surface,
            "base_bounce_err_cm": m_base["median_bounce_error_cm"],
            "base_agreement_pct": m_base["call_agreement_pct"],
            "clear_ball_f1": m_clear["ball_f1_5px"],
            "clear_bounce_err_cm": m_clear["median_bounce_error_cm"],
            "clear_agreement_pct": m_clear["call_agreement_pct"],
        })

    print(json.dumps(results_table, indent=2))